# Colab Train: BBDM Paired PNG Virtual Staining

Notebook này train mô hình BBDM cho bài toán **virtual staining** từ ảnh unstained sang ảnh H&E stained.

| | |
|---|---|
| **Input** | Ảnh unstained (grayscale), 512×512 |
| **Target** | Ảnh H&E stained (RGB), 512×512 |
| **Training crop** | 256×256 |
| **Dataset** | [mezeidragos-lateral/bach-breast-histopathology-he-staining-patches-512x512](https://huggingface.co/datasets/mezeidragos-lateral/bach-breast-histopathology-he-staining-patches-512x512) |

Notebook dùng **2 split** dữ liệu:

- `train/input` và `train/target` — để **train** model
- `test/input` và `test/target` — để **validation/visualize**

> **Hướng dẫn**: Chạy từng cell từ trên xuống. Không cần upload file lên Drive.

In [ ]:
# ✅ Bước 1: Kiểm tra GPU
# Vào Runtime → Change runtime type → GPU trước khi chạy
!nvidia-smi

## ⚙️ Cấu Hình

Chỉ cần chỉnh các tham số dưới đây nếu muốn thay đổi cấu hình mặc định.

In [ ]:
from pathlib import Path

# ── Dataset ──────────────────────────────────────────────────
HF_DATASET_ID = "mezeidragos-lateral/bach-breast-histopathology-he-staining-patches-512x512"
MAX_SAMPLES   = None         # None = dùng toàn bộ; đặt số (vd: 200) để chạy nhanh
TRAIN_RATIO   = 0.8          # 80% train, 20% test/validation

# ── Lọc ảnh có vấn đề ─────────────────────────────────────────
BLANK_THRESHOLD = 230        # Loại bỏ ảnh target có mean pixel >= ngưỡng này (gần trắng)

# ── Paths ─────────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/hongphuoc6104/bbdm_paired_virtual_staining.git"
WORKDIR       = Path('/content/bbdm_paired_virtual_staining')
DATA_ROOT     = Path('/content/data')

# ── Lưu checkpoint về Drive (QUAN TRỌNG cho long train) ─────
SAVE_TO_DRIVE    = True       # True = lưu trọng số trực tiếp vào Drive (khuyến nghị)
OUTPUT_DRIVE_DIR = Path('/content/drive/MyDrive/bbdm_outputs')

# ── Hyperparameters ───────────────────────────────────────────
CROP_SIZE  = 256
BATCH_SIZE = 1

print('HF Dataset      :', HF_DATASET_ID)
print('Train ratio     :', TRAIN_RATIO)
print('Max samples     :', MAX_SAMPLES or 'ALL')
print('Blank threshold :', BLANK_THRESHOLD)
print('Save to Drive   :', SAVE_TO_DRIVE)

## 📦 Cài Đặt Dependencies

In [ ]:
!apt-get -qq update
!apt-get -qq install -y libopenmpi-dev openmpi-bin git
%pip -q install datasets huggingface_hub blobfile mpi4py pytorch-wavelets lpips scikit-image opencv-python tqdm

## 📂 Clone Project từ GitHub

In [ ]:
import os, shutil, sys

# Reset CWD về /content để tránh lỗi khi chạy lại cell
os.chdir('/content')

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

!git clone {GITHUB_REPO} {WORKDIR}

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
print('✅ Thư mục hiện tại:', os.getcwd())

## 🤗 Tải Dataset từ HuggingFace & Chia Train/Test

- `train/input` & `train/target` — dùng để **train** model
- `test/input` & `test/target` — dùng để **validation** & hiển thị kết quả

> **Lọc ảnh**: Ảnh target gần trắng (mean pixel ≥ 230) sẽ bị loại bỏ vì là vùng mô trống, không có giá trị cho training.

In [ ]:
import io
import random
import numpy as np
from pathlib import Path
from PIL import Image

TRAIN_INPUT_DIR  = DATA_ROOT / 'train' / 'input'
TRAIN_TARGET_DIR = DATA_ROOT / 'train' / 'target'
TEST_INPUT_DIR   = DATA_ROOT / 'test' / 'input'
TEST_TARGET_DIR  = DATA_ROOT / 'test' / 'target'

# ── Kiểm tra dataset đã tồn tại chưa ──────────────────────────
def _count_png(d):
    return len(list(d.glob('*.png'))) if d.exists() else 0

_n_train_in  = _count_png(TRAIN_INPUT_DIR)
_n_train_tgt = _count_png(TRAIN_TARGET_DIR)
_n_test_in   = _count_png(TEST_INPUT_DIR)
_n_test_tgt  = _count_png(TEST_TARGET_DIR)

_dataset_ready = (
    _n_train_in > 0 and _n_train_in == _n_train_tgt
    and _n_test_in > 0 and _n_test_in == _n_test_tgt
)

if _dataset_ready:
    print('✅ Dataset đã tồn tại, bỏ qua tải lại.')
    print(f'   Train: {_n_train_in} cặp | Test: {_n_test_in} cặp')
else:
    from datasets import load_dataset

    for d in [TRAIN_INPUT_DIR, TRAIN_TARGET_DIR, TEST_INPUT_DIR, TEST_TARGET_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    print(f'📥 Đang tải dataset: {HF_DATASET_ID}...')
    ds_train = load_dataset(HF_DATASET_ID, split='train')
    try:
        ds_test_hf = load_dataset(HF_DATASET_ID, split='test')
        print(f'  HF train: {len(ds_train)} | HF test: {len(ds_test_hf)}')
    except:
        ds_test_hf = None
        print(f'  HF train: {len(ds_train)} (không có test split trên HF)')

    all_samples = list(ds_train)
    if ds_test_hf is not None:
        all_samples.extend(list(ds_test_hf))
    if MAX_SAMPLES:
        all_samples = all_samples[:MAX_SAMPLES]

    def to_pil(img_field):
        if isinstance(img_field, Image.Image):
            return img_field
        if isinstance(img_field, dict) and 'bytes' in img_field:
            return Image.open(io.BytesIO(img_field['bytes']))
        return img_field

    # ── Lọc ảnh có vấn đề ──────────────────────────────────────
    print(f'\n🔍 Lọc ảnh target gần trắng (mean ≥ {BLANK_THRESHOLD})...')
    n_before = len(all_samples)
    filtered_samples = []
    n_skipped = 0
    for sample in all_samples:
        tgt = to_pil(sample['stained_image']).convert('RGB')
        if np.mean(np.array(tgt)) >= BLANK_THRESHOLD:
            n_skipped += 1
        else:
            filtered_samples.append(sample)
    all_samples = filtered_samples
    print(f'  Trước lọc: {n_before} | Loại bỏ: {n_skipped} | Còn lại: {len(all_samples)}')

    # ── Chia train/test ────────────────────────────────────────
    random.seed(42)
    random.shuffle(all_samples)
    n_train = int(len(all_samples) * TRAIN_RATIO)
    train_samples = all_samples[:n_train]
    test_samples  = all_samples[n_train:]

    print(f'📊 Chia: {len(train_samples)} train / {len(test_samples)} test')

    def save_split(samples, input_dir, target_dir, split_name):
        for i, sample in enumerate(samples):
            fname = f'{i:05d}.png'
            to_pil(sample['unstained_image']).convert('L').save(input_dir / fname)
            to_pil(sample['stained_image']).convert('RGB').save(target_dir / fname)
            if (i + 1) % 200 == 0 or (i + 1) == len(samples):
                print(f'  [{split_name}] {i+1}/{len(samples)}')

    print('💾 Lưu ảnh train...')
    save_split(train_samples, TRAIN_INPUT_DIR, TRAIN_TARGET_DIR, 'train')
    print('💾 Lưu ảnh test...')
    save_split(test_samples, TEST_INPUT_DIR, TEST_TARGET_DIR, 'test')
    print(f'\n✅ Train: {len(list(TRAIN_INPUT_DIR.glob("*.png")))} | Test: {len(list(TEST_INPUT_DIR.glob("*.png")))}')


## 🔍 Kiểm Tra Dataset

In [ ]:
from PIL import Image as PILImage

def inspect_split(split_name, input_dir, target_dir):
    input_files = sorted(input_dir.glob('*.png'))
    target_files = sorted(target_dir.glob('*.png'))
    common = sorted({p.name for p in input_files} & {p.name for p in target_files})
    print(f'\n--- {split_name.upper()} --- input: {len(input_files)} | target: {len(target_files)} | paired: {len(common)}')
    assert common, f'Không tìm thấy cặp nào trong {split_name}!'
    for name in common[:2]:
        inp = PILImage.open(input_dir / name)
        tgt = PILImage.open(target_dir / name)
        print(f'  {name} | input {inp.mode} {inp.size} | target {tgt.mode} {tgt.size}')

inspect_split('train', TRAIN_INPUT_DIR, TRAIN_TARGET_DIR)
inspect_split('test',  TEST_INPUT_DIR,  TEST_TARGET_DIR)

## 🧪 Smoke Test: Loader & Condition Encoder

In [ ]:
import torch
from improved_diffusion.image_datasets import load_paired_png_data
from improved_diffusion.unet import ConditionEncoder

data = load_paired_png_data(
    input_dir=str(TRAIN_INPUT_DIR), target_dir=str(TRAIN_TARGET_DIR),
    batch_size=2, image_size=CROP_SIZE, deterministic=True,
)
target, cond, kwargs = next(data)
print('target:', tuple(target.shape), '| cond:', tuple(cond.shape))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = ConditionEncoder(1, 3).to(device).eval()
with torch.no_grad():
    encoded = encoder(cond.to(device).float())
print('encoded:', tuple(encoded.shape))
assert target.shape[1:] == encoded.shape[1:]
print('\n✅ OK!')

## 🚂 Smoke Train (20 steps)

Train dùng split `train`; validation dùng split `test`. Mục tiêu: kiểm tra pipeline chạy đúng.

In [ ]:
import subprocess

SMOKE_MODEL_DIR = Path('/content/bbdm_smoke_models')
SMOKE_LOG_DIR   = Path('/content/bbdm_smoke_log')
for path in [SMOKE_MODEL_DIR, SMOKE_LOG_DIR]:
    if path.exists():
        shutil.rmtree(path)

cmd = [
    sys.executable, 'train.py',
    '--lr_data_dir',     str(TRAIN_INPUT_DIR),
    '--hr_data_dir',     str(TRAIN_TARGET_DIR),
    '--val_lr_data_dir', str(TEST_INPUT_DIR),
    '--val_hr_data_dir', str(TEST_TARGET_DIR),
    '--batch_size', str(BATCH_SIZE), '--large_size', str(CROP_SIZE), '--small_size', str(CROP_SIZE),
    '--num_channels', '32', '--num_res_blocks', '1',
    '--diffusion_steps', '20', '--lr_anneal_steps', '20',
    '--log_interval', '1', '--val_interval', '5', '--save_interval', '10',
    '--model_dir', str(SMOKE_MODEL_DIR), '--log_dir', str(SMOKE_LOG_DIR),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 🖼️ Hiển Thị Kết Quả Validation (từ tập test)

In [ ]:
import matplotlib.pyplot as plt

def show_validation_triplets(log_dir, max_items=4):
    log_dir = Path(log_dir)
    triplets = []
    for idx in range(max_items):
        lr, sr, hr = log_dir / f'{idx}_LR.png', log_dir / f'{idx}_SR.png', log_dir / f'{idx}_HR.png'
        if lr.exists() and sr.exists() and hr.exists():
            triplets.append((lr, sr, hr))
    if not triplets:
        print('Không tìm thấy ảnh validation trong', log_dir)
        return
    fig, axes = plt.subplots(len(triplets), 3, figsize=(12, 4 * len(triplets)))
    if len(triplets) == 1:
        axes = [axes]
    for row, (lr, sr, hr) in zip(axes, triplets):
        for ax, path, title in zip(row, [lr, sr, hr], ['Condition (Input)', 'Prediction', 'Target']):
            ax.imshow(PILImage.open(path))
            ax.set_title(title, fontsize=13)
            ax.axis('off')
    plt.tight_layout()
    plt.show()

show_validation_triplets(SMOKE_LOG_DIR)

## 🏋️ Train Dài Hơn

**Chỉ chạy sau khi Smoke Train thành công.**

- Nếu `SAVE_TO_DRIVE = True`: checkpoint lưu **trực tiếp vào Drive** mỗi 1000 bước → không mất khi Colab ngắt kết nối
- Nếu `SAVE_TO_DRIVE = False`: checkpoint chỉ ở Colab runtime → **mất kết nối = mất toàn bộ trọng số**

> ⚠️ **Khuyến nghị**: Luôn bật `SAVE_TO_DRIVE = True` khi train dài.

In [ ]:
RUN_LONG_TRAIN = False   # ← Đổi thành True để chạy
LONG_STEPS     = 5000

if RUN_LONG_TRAIN:
    if SAVE_TO_DRIVE:
        from google.colab import drive
        from datetime import datetime
        drive.mount('/content/drive')
        run_name = f'bbdm_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
        run_root = OUTPUT_DRIVE_DIR / 'bbdm_runs' / run_name
        LONG_MODEL_DIR = run_root / 'models'
        LONG_LOG_DIR   = run_root / 'log'
        print(f'💾 Checkpoint lưu trực tiếp vào Drive:')
        print(f'   Model: {LONG_MODEL_DIR}')
        print(f'   Log  : {LONG_LOG_DIR}')
    else:
        LONG_MODEL_DIR = Path('/content/bbdm_long_models')
        LONG_LOG_DIR   = Path('/content/bbdm_long_log')
        print('⚠️  Checkpoint chỉ ở Colab runtime — sẽ mất nếu bị ngắt kết nối!')

    LONG_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    LONG_LOG_DIR.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable, 'train.py',
        '--lr_data_dir',     str(TRAIN_INPUT_DIR),
        '--hr_data_dir',     str(TRAIN_TARGET_DIR),
        '--val_lr_data_dir', str(TEST_INPUT_DIR),
        '--val_hr_data_dir', str(TEST_TARGET_DIR),
        '--batch_size', str(BATCH_SIZE), '--large_size', str(CROP_SIZE), '--small_size', str(CROP_SIZE),
        '--num_channels', '128', '--num_res_blocks', '2',
        '--diffusion_steps', '1000', '--lr_anneal_steps', str(LONG_STEPS),
        '--log_interval', '10', '--val_interval', '500', '--save_interval', '1000',
        '--model_dir', str(LONG_MODEL_DIR), '--log_dir', str(LONG_LOG_DIR),
    ]
    print('\nCommand:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('⏭️  Long train bỏ qua. Đổi RUN_LONG_TRAIN = True để chạy.')